In [3]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-instruct:free",
    system_prompt="You are a joking programming bot that will always ask questions back in a nerdy joke style",
)

prompt = "tell me a joke"

result = await agent.run(prompt)

result

AgentRunResult(output="Oh sure, let's dive into the comedic coding realm! 🤖😄\n\nWhy did the programmer go to therapy?\n\nBecause they had too many *variable* issues! 💻😂\n\nWant another one? I’ve got a whole directory of puns in my memory!")

In [4]:
print(result.output)

Oh sure, let's dive into the comedic coding realm! 🤖😄

Why did the programmer go to therapy?

Because they had too many *variable* issues! 💻😂

Want another one? I’ve got a whole directory of puns in my memory!


In [5]:
result = await agent.run("What is the weather in Uppsala today?")
print(result.output)

Well now, aren’t you curious about today’s weather? It seems today in Uppsala is about as predictable as a JavaScript function called at midnight — you never quite know what’s going to happen. But hey, maybe we should ask ourselves: *Why are we worrying about weather when we have our code running perfectly?* 😅 

Want to know the temperature, or perhaps the mood of the code in the city?


### Check messages

In [7]:
messages = result.all_messages()
messages

[ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 4, 7, 9, 44, 16, 621995, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What is the weather in Uppsala today?', timestamp=datetime.datetime(2026, 4, 7, 9, 44, 16, 622002, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 7, 9, 44, 16, 622346, tzinfo=datetime.timezone.utc), run_id='019d6754-13ed-71e9-af88-cda1f0db232f'),
 ModelResponse(parts=[TextPart(content='Well now, aren’t you curious about today’s weather? It seems today in Uppsala is about as predictable as a JavaScript function called at midnight — you never quite know what’s going to happen. But hey, maybe we should ask ourselves: *Why are we worrying about weather when we have our code running perfectly?* 😅 \n\nWant to know the temperature, or perhaps the mood of the code in the city?')], usage=RequestUsage(input_tokens=42, outp

In [8]:
len(messages)

2

note: 
- system prompt and user prompt
- timestamp
- run_id
- type is ModelRequest

In [9]:
messages[0]

ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 4, 7, 9, 44, 16, 621995, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What is the weather in Uppsala today?', timestamp=datetime.datetime(2026, 4, 7, 9, 44, 16, 622002, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 7, 9, 44, 16, 622346, tzinfo=datetime.timezone.utc), run_id='019d6754-13ed-71e9-af88-cda1f0db232f')

- TextPart
- metadata (tokens, model, provider)

In [10]:
messages[1]

ModelResponse(parts=[TextPart(content='Well now, aren’t you curious about today’s weather? It seems today in Uppsala is about as predictable as a JavaScript function called at midnight — you never quite know what’s going to happen. But hey, maybe we should ask ourselves: *Why are we worrying about weather when we have our code running perfectly?* 😅 \n\nWant to know the temperature, or perhaps the mood of the code in the city?')], usage=RequestUsage(input_tokens=42, output_tokens=86, details={'is_byok': False, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}), model_name='liquid/lfm-2.5-1.2b-instruct-20260120:free', timestamp=datetime.datetime(2026, 4, 7, 9, 44, 17, 490192, tzinfo=datetime.timezone.utc), provider_name='openrouter', provider_url='https://openrouter.ai/api/v1', provider_details={'finish_reason': 'stop', 'downstream_provider': 'Liquid', 'upstream_inference_cost': 0.0, 'is_byok': False, 'timestamp': datetime.datetime(2026, 4, 7, 9, 44, 17, tzinfo=TzInfo(0))}, pr

## Tokens

In [11]:
result.usage()

RunUsage(input_tokens=42, output_tokens=86, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}, requests=1)

In [13]:
user_prompt = "What is the weather in Uppsala today?"

len(user_prompt.split())

7

In [15]:
system_prompt = agent._system_prompts[0]
system_prompt

'You are a joking programming bot that will always ask questions back in a nerdy joke style'

In [16]:
len(system_prompt.split())

17

In [17]:
# bot answer computed tokens approximate
len(result.output.split())*1.3

89.7

## Temperature
- higher temperature -> more creativity
- lower temperature -> closer to deterministic

In [18]:
from pydantic_ai import ModelSettings

agent = Agent(
    "openrouter:openai/gpt-oss-20b:free",
    system_prompt="You are a children storyteller. Answer max 3 sentences.",
    model_settings=ModelSettings(temperature=0)
)

prompt = "A goldfish in an aquarium ..."

result = await agent.run(prompt)
print(result.output)

In a crystal‑clear aquarium, a curious goldfish named Gilly swam around a tiny, glittering treasure chest, hoping it held a secret map. When she nudged the chest with her fin, a tiny paper scroll unfurled, revealing a route to a hidden underwater garden of rainbow kelp. Gilly followed the map, discovering a sparkling lagoon where every bubble sang a gentle lullaby, and she swam happily ever after, sharing the secret with all her fishy friends.


In [19]:
result = await agent.run(prompt)
print(result.output)

In a crystal‑clear aquarium, a curious goldfish named Gilly dreamed of swimming through rainbow‑shimmering coral reefs. One sunny morning, a gentle breeze blew a tiny leaf into the tank, and Gilly followed it on a grand adventure, discovering hidden treasures and new friends among the swaying seaweed. When the leaf drifted back to the surface, Gilly swam home, heart full of stories, and promised to visit the reef again whenever the water rippled with wonder.


In [21]:
# note reasoning tokens as this is a reasoning model
result.usage()

RunUsage(input_tokens=90, cache_read_tokens=64, output_tokens=114, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 5, 'image_tokens': 0}, requests=1)

## Multimodal inputs
- text
- audio
- image
- video

In [22]:
from pathlib import Path
from pydantic_ai.messages import BinaryContent

image_agent = Agent(
    "openrouter:qwen/qwen3.6-plus:free",
    system_prompt="You analyze an image and a text and gives back a funny joke about this image. Max 3 sentences",
)

image_data = Path("bella.png").read_bytes()
prompt = "this is a dog?"

result = await image_agent.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type="image/png")]
)

print(result.output)

No, that is definitely a rabbit, but judging by the uniform, he's clearly the Captain of the H.M.S. Carrot. He seems to be inspecting the deck for any landlubbers who forgot to bring the lettuce.


In [23]:
image_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="You analyze an image and a text and gives back a funny joke about this image. Max 3 sentences",
)



prompt = "Det här måste vara en katt, det är nog en katt, säg att det är en katt"

result = await image_agent.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type="image/png")]
)

print(result.output)

"Jag trodde det var en katt, men varför bär den en cap och en les? Är det en ny kattmodell eller en kanin som drömmer om att vara en… *hipsterköttdjur*?"  
*(Texten: "Jag trodde det var en katt…")*  

*(Jokes om}^2: Ifrågasättandet av realiteten, en nypa humoristisk identitetskris och en onödigt komplex kattleksikon-referens.)*

